In [3]:
# Run this in a cell to downgrade chromadb
!pip install chromadb==0.4.24

  Using cached chromadb-0.4.24-py3-none-any.whl.metadata (7.3 kB)
  Using cached chroma-hnswlib-0.7.3.tar.gz (31 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pulsar_client-3.8.0-cp313-cp313-win_amd64.whl.metadata (1.2 kB)
  Using cached opentelemetry_instrumentation_fastapi-0.59b0-py3-none-any.whl.metadata (2.2 kB)
  Using cached opentelemetry_instrumentation_asgi-0.59b0-py3-none-any.whl.metadata (2.0 kB)
  Using cached opentelemetry_instrumentation-0.59b0-py3-none-any.whl.metadata (7.1 kB)
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-fastapi to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemet

  error: subprocess-exited-with-error
  
  × Building wheel for chroma-hnswlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [5 lines of output]
      running bdist_wheel
      running build
      running build_ext
      building 'hnswlib' extension
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for chroma-hnswlib

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Failed to build installable wheels for some pyproject.toml based projects (chroma-hnswlib)


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [7]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [8]:
books["tagged_description"].to_string("tagged_description.txt", index=False)

In [9]:
raw_documents = TextLoader("tagged_description.txt", encoding='utf-8').load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [11]:
from langchain.schema import Document

# Create documents directly from the DataFrame - one document per book
documents = []
for _, row in books.iterrows():
    doc = Document(
        page_content=row['description'],  # Just the description, not ISBN
        metadata={
            'isbn13': int(row['isbn13']),  # Store ISBN as metadata
            'title': row['title'],
            'authors': row['authors'],
            'categories': row['categories']
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")


Created 5197 documents


In [12]:
db_books = Chroma.from_documents(
    documents,
    embedding=OpenAIEmbeddings()) # Try 'embedding' explicitly, just in case the function expects this)

In [13]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='7ff788ca-78a1-449c-8a19-db3f1e805886', metadata={'categories': 'Juvenile Fiction', 'authors': 'Marilyn Singer;Julie Aigner-Clark', 'title': 'Baby Einstein: Neighborhood Animals', 'isbn13': 9780786808069}, page_content='Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='5438169e-93b5-4102-b014-659f8a073a56', metadata={'categories': 'Juvenile Fiction', 'title': 'Baby Einstein: Babies', 'isbn13': 9780786808380, 'authors': 'Julie Aigner-Clark'}, page_content="Introduce your babies to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies eac

In [14]:
books[books["isbn13"] == docs[0].metadata['isbn13']]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [15]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    # Perform similarity search
    recs = db_books.similarity_search(query, k=top_k)

    # Extract ISBNs from metadata
    books_list = [rec.metadata['isbn13'] for rec in recs]

    # Filter books and preserve order
    result_books = books[books["isbn13"].isin(books_list)]

    # Maintain the ranking order from similarity search
    isbn_to_rank = {isbn: i for i, isbn in enumerate(books_list)}
    result_books = result_books.copy()
    result_books['rank'] = result_books['isbn13'].map(isbn_to_rank)
    result_books = result_books.sort_values('rank').drop('rank', axis=1)

    return result_books.head(top_k)


In [16]:
retrieve_semantic_recommendations("A book to teach children about science")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...
2887,9780618056736,0618056734,Unweaving the Rainbow,Richard Dawkins,Science,http://books.google.com/books/content?id=GWj0q...,"Offers an assessment of what science is, how i...",2000.0,4.02,352.0,9168.0,"Unweaving the Rainbow: Science, Delusion and t...",9780618056736 Offers an assessment of what sci...
1351,9780316281331,0316281336,"The World Treasury of Physics, Astronomy, and ...",Timothy Ferris;Clifton Fadiman,Education,http://books.google.com/books/content?id=zryXG...,The writings of more than 60 authors including...,1993.0,4.12,859.0,361.0,"The World Treasury of Physics, Astronomy, and ...",9780316281331 The writings of more than 60 aut...
2658,9780553103748,0553103741,The Illustrated a Brief History of Time,Stephen Hawking,Science,http://books.google.com/books/content?id=wDr8u...,"An illustrated, large-format edition of the be...",1996.0,4.16,256.0,750.0,The Illustrated a Brief History of Time,"9780553103748 An illustrated, large-format edi..."
2477,9780471265184,0471265187,Astronomy,Dinah L. Moché,Science,http://books.google.com/books/content?id=vBFuF...,"""A lively, up-to-date account of the basic pri...",2004.0,3.89,343.0,109.0,Astronomy: A Self-Teaching Guide,"9780471265184 ""A lively, up-to-date account of..."
549,9780131871656,013187165X,Astronomy,Eric Chaisson;Stephen McMillan,Mathematics,http://books.google.com/books/content?id=1O00A...,This introduction to astronomy features an exc...,2006.0,3.85,499.0,153.0,Astronomy: a beginner's guide to the universe,9780131871656 This introduction to astronomy f...
3767,9780786849734,0786849738,Disney's Little Einsteins: Galactic Goodnight,Susan Ring,Juvenile Fiction,http://books.google.com/books/content?id=Nm3N3...,"The Little Einsteins, equipped with their came...",2006.0,4.50,20.0,10.0,Disney's Little Einsteins: Galactic Goodnight,"9780786849734 The Little Einsteins, equipped w..."
4383,9781400077809,140007780X,Einstein's Dreams,Alan P. Lightman,Fiction,http://books.google.com/books/content?id=1oSGm...,Young Albert Einstein's dreams are imagined in...,1993.0,4.07,144.0,24345.0,Einstein's Dreams,9781400077809 Young Albert Einstein's dreams a...
2409,9780451456816,0451456815,Not of Woman Born,Constance Ash;Robert A. Silverberg,Fiction,http://books.google.com/books/content?id=nOEIA...,A collection of stories that detail modern sci...,1999.0,3.80,288.0,59.0,Not of Woman Born,9780451456816 A collection of stories that det...
3579,9780762427321,0762427329,The Illustrated on the Shoulders of Giants,Stephen W. Hawking,Science,http://books.google.com/books/content?id=r1y3P...,Hawking's revolutionary look at the scientific...,2005.0,4.20,256.0,4014.0,The Illustrated on the Shoulders of Giants: Th...,9780762427321 Hawking's revolutionary look at ...
